# Capstone Three: Data Wrangling and Exploratory Data Analysis

## Business Problem
The goal of this project is to analyze supply chain operations to identify the key factors driving higher costs and delivery delays, with the aim of improving efficiency and optimizing overall performance.

This notebook focuses on preparing the raw dataset for analysis by performing data wrangling tasks such as assessing data quality, handling missing values, removing duplicates, identifying outliers, correcting data types, and applying necessary transformations.

After cleaning the dataset, exploratory data analysis will be conducted to identify important patterns, relationships, and trends in the data.

## Project Objective
The objective of this analysis is to answer the following question:

Which key factors in the supply chain are driving higher costs and delivery delays, and what data-driven actions can improve efficiency?

## Deliverables
This notebook documents the full data wrangling and exploratory data analysis process. A cleaned version of the dataset will also be saved as a separate file for use in future analysis and modeling.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams['figure.figsize'] = (10, 6)
sns.set()

## Imports and Setup
The necessary Python libraries were imported for data manipulation, visualization, and basic file handling. Display settings were also adjusted to make the dataset easier to inspect during the wrangling process.

## Load the Raw Data
In this section, the raw dataset is loaded into a pandas DataFrame. The first step in data wrangling is to inspect the dataset structure, size, and general contents before making any cleaning decisions.

In [2]:
df = pd.read_csv("supply_chain_data.csv")

In [3]:
df.head()

,Product type,SKU,Price,Availability,Number of products sold,Revenue generated,Customer demographics,Stock levels,Lead times,Order quantities,Shipping times,Shipping carriers,Shipping costs,Supplier name,Location,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Inspection results,Defect rates,Transportation modes,Routes,Costs
0,haircare,SKU0,69.81,55,802,8662.00,Non-binary,58,7,96,4,Carrier B,2.96,Supplier 3,Mumbai,29,215,29,46.28,Pending,0.23,Road,Route B,187.75
1,skincare,SKU1,14.84,95,736,7460.90,Female,53,30,37,2,Carrier A,9.72,Supplier 3,Mumbai,23,517,30,33.62,Pending,4.85,Road,Route B,503.07
2,haircare,SKU2,11.32,34,8,9577.75,Unknown,1,10,88,2,Carrier B,8.05,Supplier 1,Mumbai,12,971,27,30.69,Pending,4.58,Air,Route C,141.92
3,skincare,SKU3,61.16,68,83,7766.84,Non-binary,23,13,59,6,Carrier C,1.73,Supplier 5,Kolkata,24,937,18,35.62,Fail,4.75,Rail,Route A,254.78
4,skincare,SKU4,4.81,26,871,2686.51,Non-binary,5,3,56,8,Carrier A,3.89,Supplier 1,Delhi,5,414,3,92.07,Fail,3.15,Air,Route A,923.44


In [4]:
df.shape

(100, 24)

In [5]:
df.columns

Index(['Product type', 'SKU', 'Price', 'Availability',
       'Number of products sold', 'Revenue generated', 'Customer demographics',
       'Stock levels', 'Lead times', 'Order quantities', 'Shipping times',
       'Shipping carriers', 'Shipping costs', 'Supplier name', 'Location',
       'Lead time', 'Production volumes', 'Manufacturing lead time',
       'Manufacturing costs', 'Inspection results', 'Defect rates',
       'Transportation modes', 'Routes', 'Costs'],
      dtype='object')

The dataset was successfully loaded into the notebook. It contains 100 records and 24 variables. Initial inspection confirms that the dataset includes a mix of numerical and categorical features relevant to supply chain cost, delivery performance, and operational efficiency.

## Initial Inspection
This section provides a first look at the structure and quality of the dataset. This includes checking data types, identifying missing values, reviewing summary statistics, and looking for possible inconsistencies.

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Product type             100 non-null    object 
 1   SKU                      100 non-null    object 
 2   Price                    100 non-null    float64
 3   Availability             100 non-null    int64  
 4   Number of products sold  100 non-null    int64  
 5   Revenue generated        100 non-null    float64
 6   Customer demographics    100 non-null    object 
 7   Stock levels             100 non-null    int64  
 8   Lead times               100 non-null    int64  
 9   Order quantities         100 non-null    int64  
 10  Shipping times           100 non-null    int64  
 11  Shipping carriers        100 non-null    object 
 12  Shipping costs           100 non-null    float64
 13  Supplier name            100 non-null    object 
 14  Location                 10

## Variable Notes

- product_type: Type of product in the supply chain
- sku: Unique stock keeping unit identifier for each item
- price: Selling price of the product
- availability: Units available in stock
- number_of_products_sold: Total units sold
- revenue_generated: Revenue earned from product sales
- stock_levels: Inventory available for the product
- lead_times: Time required before shipment or fulfillment
- shipping_times: Time taken to deliver the order
- shipping_costs: Cost associated with shipping
- supplier_name: Supplier responsible for providing the product
- location: Operational or shipment location
- production_volumes: Number of units produced
- manufacturing_costs: Cost of manufacturing the product
- inspection_results: Outcome of quality inspection
- defect_rates: Share of products with defects
- transportation_modes: Method of transportation used
- routes: Delivery route used
- costs: Overall cost metric used in the dataset

In [7]:
df.describe()

,Price,Availability,Number of products sold,Revenue generated,Stock levels,Lead times,Order quantities,Shipping times,Shipping costs,Lead time,Production volumes,Manufacturing lead time,Manufacturing costs,Defect rates,Costs
count,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
mean,49.46,48.40,460.99,5776.05,47.77,15.96,49.22,5.75,5.55,17.08,567.84,14.77,47.27,2.28,529.25
std,31.17,30.74,303.78,2732.84,31.37,8.79,26.78,2.72,2.65,8.85,263.05,8.91,28.98,1.46,258.30
min,1.70,1.00,8.00,1061.62,0.00,1.00,1.00,1.00,1.01,1.00,104.00,1.00,1.09,0.02,103.92
25%,19.60,22.75,184.25,2812.85,16.75,8.00,26.00,3.75,3.54,10.00,352.00,7.00,22.98,1.01,318.78
50%,51.24,43.50,392.50,6006.35,47.50,17.00,52.00,6.00,5.32,18.00,568.50,14.00,45.91,2.14,520.43
75%,77.20,75.00,704.25,8253.98,73.00,24.00,71.25,8.00,7.60,25.00,797.00,23.00,68.62,3.56,763.08
max,99.17,100.00,996.00,9866.47,100.00,30.00,96.00,10.00,9.93,30.00,985.00,30.00,99.47,4.94,997.41


In [8]:
df.describe(include='object')

,Product type,SKU,Customer demographics,Shipping carriers,Supplier name,Location,Inspection results,Transportation modes,Routes
count,100,100,100,100,100,100,100,100,100
unique,3,100,4,3,5,5,3,4,3
top,skincare,SKU0,Unknown,Carrier B,Supplier 1,Kolkata,Pending,Road,Route A
freq,40,1,31,43,27,25,41,29,43


In [9]:
df.isnull().sum().sort_values(ascending=False)

Product type               0
SKU                        0
Routes                     0
Transportation modes       0
Defect rates               0
Inspection results         0
Manufacturing costs        0
Manufacturing lead time    0
Production volumes         0
Lead time                  0
Location                   0
Supplier name              0
Shipping costs             0
Shipping carriers          0
Shipping times             0
Order quantities           0
Lead times                 0
Stock levels               0
Customer demographics      0
Revenue generated          0
Number of products sold    0
Availability               0
Price                      0
Costs                      0
dtype: int64

The dataset was checked for missing values, and no missing values were found across any of the 24 variables. Therefore, no imputation or removal of rows or columns was necessary.

## Data Organization and Definition
Before cleaning the data, it is important to understand what each variable represents and whether the data types are appropriate.

Below is a review of the dataset columns, their meanings, and whether they are numeric, categorical, text, or date related variables.

In [10]:
column_summary = pd.DataFrame({
    "column_name": df.columns,
    "data_type": df.dtypes.values,
    "missing_values": df.isnull().sum().values,
    "unique_values": [df[col].nunique() for col in df.columns]
})

column_summary

,column_name,data_type,missing_values,unique_values
0,Product type,object,0,3
1,SKU,object,0,100
2,Price,float64,0,100
3,Availability,int64,0,63
4,Number of products sold,int64,0,96
5,Revenue generated,float64,0,100
6,Customer demographics,object,0,4
7,Stock levels,int64,0,65
8,Lead times,int64,0,29
9,Order quantities,int64,0,61


Based on the summary above, the dataset includes a mix of numeric and categorical features. At this stage, I reviewed the variables to determine which columns may require type conversion, renaming, or further cleaning before analysis.

### Variable Notes

- product_type: Category of product being shipped  
- shipping_costs: Cost associated with shipping the product  
- manufacturing_costs: Cost of producing the product  
- costs: Total operational cost related to the shipment  
- revenue_generated: Revenue generated from the shipment  
- lead_times: Time taken for production or processing  
- shipping_times: Time taken for delivery  
- defect_rates: Percentage of defective products  
- location: Geographic location of operations  
- shipping_carriers: Company responsible for delivery  
- transportation_modes: Mode of transport used (air, road, etc.)  

## Basic Cleaning and Formatting
To improve readability and consistency, the column names were standardized. Text values were also stripped of extra spaces where needed.

In [11]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(r'[^\w]', '', regex=True)
df.columns

Index(['product_type', 'sku', 'price', 'availability',
       'number_of_products_sold', 'revenue_generated', 'customer_demographics',
       'stock_levels', 'lead_times', 'order_quantities', 'shipping_times',
       'shipping_carriers', 'shipping_costs', 'supplier_name', 'location',
       'lead_time', 'production_volumes', 'manufacturing_lead_time',
       'manufacturing_costs', 'inspection_results', 'defect_rates',
       'transportation_modes', 'routes', 'costs'],
      dtype='object')

In [12]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()

Standardizing column names makes the dataset easier to work with and reduces errors later in the analysis. This also helps maintain a consistent coding style throughout the notebook.

## Missing Values
Missing values can affect the reliability of the analysis, so each variable was checked for null or missing entries. The decision to drop, keep, or impute missing values depends on the amount of missingness and the role of the variable in the business problem.

In [13]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_percent
}).sort_values(by="missing_percent", ascending=False)

missing_table[missing_table["missing_count"] > 0]

,missing_count,missing_percent


The table above shows which variables contain missing values and the percentage missing in each. Variables with a very high percentage of missing data may need to be dropped, while variables with a small percentage of missing values may be imputed or have rows removed depending on the situation.

In [14]:
threshold = 50
cols_to_drop = missing_table[missing_table["missing_percent"] > threshold].index.tolist()
cols_to_drop

[]

In [15]:
df = df.drop(columns=cols_to_drop)

In [16]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [17]:
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    df[col] = df[col].replace('nan', np.nan)
    df[col] = df[col].fillna('Unknown')

The dataset was checked for missing values, and no missing values were found across any of the variables. Therefore, no imputation or removal of rows or columns was required.

## Duplicate Records
Duplicate observations can bias summary statistics and distort analysis results. In this section, duplicate rows are identified and removed where appropriate.

In [18]:
df.duplicated().sum()

0

In [19]:
duplicates = df[df.duplicated()]
duplicates.head()

,product_type,sku,price,availability,number_of_products_sold,revenue_generated,customer_demographics,stock_levels,lead_times,order_quantities,shipping_times,shipping_carriers,shipping_costs,supplier_name,location,lead_time,production_volumes,manufacturing_lead_time,manufacturing_costs,inspection_results,defect_rates,transportation_modes,routes,costs


In [20]:
df.duplicated().sum()
df['sku'].duplicated().sum()

0

Duplicate records were checked at both the full row level and through the SKU field, which appears to uniquely identify each product record in this dataset. No duplicate records were found, so no duplicate rows were removed.

## Data Type Corrections
Some variables may not be stored in the correct format. For example, date columns may be stored as text, or numeric variables may be stored as strings. Correcting data types is necessary before analysis.

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 24 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   product_type             100 non-null    object 
 1   sku                      100 non-null    object 
 2   price                    100 non-null    float64
 3   availability             100 non-null    int64  
 4   number_of_products_sold  100 non-null    int64  
 5   revenue_generated        100 non-null    float64
 6   customer_demographics    100 non-null    object 
 7   stock_levels             100 non-null    int64  
 8   lead_times               100 non-null    int64  
 9   order_quantities         100 non-null    int64  
 10  shipping_times           100 non-null    int64  
 11  shipping_carriers        100 non-null    object 
 12  shipping_costs           100 non-null    float64
 13  supplier_name            100 non-null    object 
 14  location                 10

Data types were reviewed using the dataset information. The majority of variables were already in appropriate formats, so no major data type conversions were required.

## Data Quality Checks
Data quality checks were performed to identify unrealistic or inconsistent values in key variables such as costs, defect rates, and operational metrics.

In [22]:
df[['costs', 'shipping_costs', 'manufacturing_costs', 'defect_rates']].describe()

df[df['costs'] < 0]
df[df['shipping_costs'] < 0]
df[df['manufacturing_costs'] < 0]

df[(df['defect_rates'] < 0) | (df['defect_rates'] > 100)]

,product_type,sku,price,availability,number_of_products_sold,revenue_generated,customer_demographics,stock_levels,lead_times,order_quantities,shipping_times,shipping_carriers,shipping_costs,supplier_name,location,lead_time,production_volumes,manufacturing_lead_time,manufacturing_costs,inspection_results,defect_rates,transportation_modes,routes,costs


The data quality checks did not identify any negative cost values or impossible defect rate values. This suggests that the main numerical fields are internally consistent and suitable for further analysis.

## Outlier Detection
Outliers were examined to determine whether they represent genuine extreme cases or data quality problems. Outliers were not removed automatically. Instead, they were reviewed in the context of the variable and business problem.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for col in ['costs', 'shipping_costs', 'manufacturing_costs', 'revenue_generated', 'defect_rates']:
    plt.figure()
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()

Outliers were reviewed in key cost and performance variables using boxplots. Because supply chain data can naturally include extreme but valid operational cases, these values were examined but not removed automatically.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
numeric_cols

In [ ]:
sns.boxplot(x=df['revenue_generated'])
plt.title('Boxplot of Revenue Generated')
plt.show()

In [ ]:
Q1 = df['revenue_generated'].quantile(0.25)
Q3 = df['revenue_generated'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['revenue_generated'] < lower_bound) | (df['revenue_generated'] > upper_bound)]
outliers.shape

Outliers were identified using the interquartile range method and visualized using boxplots. These values were reviewed before making any decision. Depending on the context, extreme values may reflect real behavior rather than data errors, so they were handled cautiously.

In [ ]:
df['revenue_generated'] = np.where(
    df['revenue_generated'] > upper_bound,
    upper_bound,
    df['revenue_generated']
)

df['revenue_generated'] = np.where(
    df['revenue_generated'] < lower_bound,
    lower_bound,
    df['revenue_generated']
)

## Additional Transformations
After the initial cleaning steps, additional transformations were applied to improve the usability of the dataset for exploratory analysis.

In [ ]:
if 'gender' in df.columns:
    df['gender'] = df['gender'].replace({
        'M': 'Male',
        'F': 'Female',
        'male': 'Male',
        'female': 'Female'
    })

In [ ]:
if 'date' in df.columns:
    df['year'] = df['date'].dt.year
    df['month'] = df['date'].dt.month

In [ ]:
if 'age' in df.columns:
    df['age_group'] = pd.cut(
        df['age'],
        bins=[0, 18, 35, 50, 65, 100],
        labels=['0 to 18', '19 to 35', '36 to 50', '51 to 65', '66 plus']
    )

These transformations were applied to make the dataset more consistent and more informative for analysis. Standardized labels help reduce category duplication, while derived variables may support better insight generation during EDA.

## Final Data Check
After completing the wrangling process, the cleaned dataset was reviewed one final time to confirm that major quality issues had been addressed.

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False)

In [ ]:
df.duplicated().sum()

In [ ]:
df.head()

The final review confirms that the dataset is now in a cleaner and more analysis ready state. Missing values, duplicates, formatting issues, and other quality concerns have been addressed where appropriate.

## Save the Cleaned Dataset
After data cleaning was completed, the cleaned dataset was saved as a CSV file so it can be used in future stages of the capstone project.

In [ ]:
output_path = "cleaned_dataset.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved as: {output_path}")

## Exploratory Data Analysis
With the cleaned dataset prepared, exploratory data analysis was conducted to better understand the distributions, trends, and relationships within the data.

In [ ]:
df.describe()

In [ ]:
df.describe(include='object')

The summary statistics provide an overview of the central tendency, spread, and frequency patterns in the dataset. These values help identify broad trends and support the interpretation of the visualizations that follow.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    plt.figure()
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

The distribution plots reveal the shape of each numeric variable, including whether the data are skewed, approximately normal, or affected by extreme values. These patterns are useful when deciding how to interpret or model the variables later.

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns

for col in categorical_cols:
    plt.figure()
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f'Count Plot of {col}')
    plt.xticks(rotation=45)
    plt.show()

The categorical count plots show the frequency distribution of the main categories in the dataset. This helps identify dominant groups, possible imbalance, and unusual or sparse categories.

## Cost and Delay Analysis
This section explores how different supply chain factors relate to costs and delivery performance. The goal is to identify the main drivers of higher costs and longer shipping times.

In [ ]:
corr = df.select_dtypes(include=np.number).corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

The correlation matrix highlights relationships between key numerical variables such as costs, shipping costs, and delivery times. Strong correlations may indicate which factors contribute most to higher operational costs or delays.

### Product Type Analysis

In [ ]:
df.groupby('product_type')[['costs', 'shipping_costs', 'shipping_times']].mean().sort_values('costs', ascending=False)

### Location Analysis

In [ ]:
df.groupby('location')[['costs', 'shipping_costs', 'shipping_times']].mean().sort_values('costs', ascending=False)

Differences across locations suggest that some regions experience higher costs and longer delivery times, possibly due to operational or logistical challenges.

### Shipping Carrier Analysis

In [ ]:
df.groupby('shipping_carriers')[['costs', 'shipping_costs', 'shipping_times']].mean().sort_values('costs', ascending=False)

Shipping carriers show variation in cost and delivery performance, indicating that carrier selection may impact both efficiency and cost.

### Transportation Mode Analysis

In [ ]:
df.groupby('transportation_modes')[['costs', 'shipping_times', 'defect_rates']].mean().sort_values('costs', ascending=False)

Transportation modes differ in both cost and delivery time, suggesting that mode selection plays a key role in optimizing supply chain performance.

This analysis shows that certain product types have higher average costs and shipping times, indicating potential inefficiencies in handling or production.

Grouped analysis was conducted across key supply chain factors such as product type, location, shipping carriers, and transportation modes. This helps identify which factors contribute most to higher costs and longer delivery times.

### Overall Insight

The analysis shows that supply chain costs and delays are influenced by multiple factors, including product type, location, shipping carriers, and transportation modes. Certain categories consistently exhibit higher costs and longer delivery times, indicating key areas where operational improvements can be made.

## Key Findings

1. The dataset was successfully cleaned and validated, with no missing values or duplicate records identified.

2. Product types vary significantly in terms of cost and delivery time, indicating that certain categories may be less efficient.

3. Location-based differences highlight that some regions incur higher operational costs and longer shipping times.

4. Shipping carriers and transportation modes show noticeable variation in performance, suggesting opportunities for optimization.

5. Defect rates and operational factors may contribute indirectly to increased costs and delays.

## Recommendations

1. Focus cost optimization efforts on high-cost product categories.
2. Investigate operational inefficiencies in locations with higher delays.
3. Optimize carrier and transportation mode selection to improve delivery efficiency and reduce costs.

## Conclusion

This analysis identified key factors contributing to higher supply chain costs and delivery delays, including product categories, geographic locations, and transportation choices. 

The findings highlight clear opportunities to improve operational efficiency by optimizing logistics strategies, selecting more efficient carriers, and focusing on high-cost areas. 

These insights provide a strong foundation for future predictive modeling and data-driven decision-making in supply chain optimization.